In [15]:
import numpy as np
import pandas as pd
import altair as alt

np.random.seed(7)

# ----------------------------
# Simulate data
# ----------------------------
n_league = 360
n_bulls = 20

usage_league = np.clip(
    np.random.normal(22, 5.5, n_league),
    8, 36
)

usage_bulls = np.clip(
    np.random.normal(23, 4.5, n_bulls),
    10, 34
)

def base_efficiency(u):
    return 1.12 - 0.012 * (u - 22)**2 / 10

eff_league = base_efficiency(usage_league) + np.random.normal(
    0, 0.07 + 0.004 * np.maximum(usage_league - 20, 0), n_league
)

eff_bulls = base_efficiency(usage_bulls) + np.random.normal(
    0, 0.06 + 0.003 * np.maximum(usage_bulls - 20, 0), n_bulls
)

df = pd.DataFrame({
    "Usage Rate": np.concatenate([usage_bulls, usage_league]),
    "Efficiency": np.concatenate([eff_bulls, eff_league]),
    "Team": ["Bulls"] * n_bulls + ["Other"] * n_league
})

df = df[
    (df["Usage Rate"] >= 8) &
    (df["Efficiency"] >= 0.5)
]


# ----------------------------
# Precompute smooth league trend
# ----------------------------
bins = np.linspace(8, 36, 30)

trend_df = (
    df[df["Team"] == "Other"]
    .assign(bin=pd.cut(df[df["Team"] == "Other"]["Usage Rate"], bins))
    .groupby("bin", observed=True)
    .agg(
        Usage=("Usage Rate", "mean"),
        Efficiency=("Efficiency", "mean")
    )
    .dropna()
)

# ----------------------------
# Scatter plot
# ----------------------------
scatter = alt.Chart(df).mark_circle(opacity=0.7).encode(
    x=alt.X("Usage Rate:Q", title="Usage Rate (%)", scale=alt.Scale(domain=[8, 36])),
    y=alt.Y("Efficiency:Q", title="Points per Possession", scale=alt.Scale(domain=[0.5, 1.35])),
    color=alt.Color(
        "Team:N",
        scale=alt.Scale(
            domain=["Bulls", "Other"],
            range=["#CE1141", "#C7C7C7"]
        ),
        legend=alt.Legend(title=None)
    ),
    tooltip=["Usage Rate", "Efficiency", "Team"]
)

# ----------------------------
# Trend line (precomputed)
# ----------------------------
trend = alt.Chart(trend_df).mark_line(
    color="black",
    strokeWidth=3
).encode(
    x="Usage:Q",
    y="Efficiency:Q"
)

optimal_band = alt.Chart(
    pd.DataFrame({
        "x1": [18],
        "x2": [24]
    })
).mark_rect(
    opacity=0.08,
    color="gray"
).encode(
    x="x1:Q",
    x2="x2:Q"
)


league_avg = alt.Chart(
    pd.DataFrame({"y": [df["Efficiency"].mean()]})
).mark_rule(
    strokeDash=[4,4],
    color="black",
    opacity=0.4
).encode(
    y="y:Q"
)

scatter = scatter.encode(
    opacity=alt.condition(
        alt.datum.Team == "Bulls",
        alt.value(1.0),
        alt.value(0.5)
    )
)



# ----------------------------
# Final chart
# ----------------------------
chart = (optimal_band + scatter + trend + league_avg).properties(
    width=680,
    height=450,
    title="Usage Rate vs Offensive Efficiency"
)


chart


alt.LayerChart(...)

In [16]:
import pandas as pd

table_df = pd.DataFrame({
    "Input": [
        "Usage Rate",
        "Offensive Efficiency",
        "Lineup Context"
    ],
    "Definition": [
        "Share of team possessions used",
        "Points per possession",
        "Teammates and role"
    ],
    "Observed Range": [
        "~8%–36%",
        "~0.8–1.3",
        "Varies"
    ],
    "Key Impact": [
        "High sensitivity",
        "Outcome metric",
        "Moderate sensitivity"
    ]
})

table_df


,Input,Definition,Observed Range,Key Impact
0,Usage Rate,Share of team possessions used,~8%–36%,High sensitivity
1,Offensive Efficiency,Points per possession,~0.8–1.3,Outcome metric
2,Lineup Context,Teammates and role,Varies,Moderate sensitivity


In [21]:
import numpy as np
import pandas as pd
import altair as alt

np.random.seed(12)

# ----------------------------
# Simulate lineup data
# ----------------------------
n_league = 260
n_bulls = 18

# Lineup stability (shared minutes)
minutes_league = np.clip(
    np.random.exponential(scale=180, size=n_league),
    10, 900
)

minutes_bulls = np.clip(
    np.random.normal(loc=420, scale=140, size=n_bulls),
    80, 900
)

# Volatility decreases with stability
def volatility(m):
    return 6.5 - 1.4 * np.log(m / 60 + 1)

vol_league = volatility(minutes_league) + np.random.normal(0, 0.6, n_league)
vol_bulls = volatility(minutes_bulls) + np.random.normal(0, 0.5, n_bulls)

df = pd.DataFrame({
    "Shared Minutes": np.concatenate([minutes_bulls, minutes_league]),
    "Net Rating Variability": np.concatenate([vol_bulls, vol_league]),
    "Team": ["Bulls"] * n_bulls + ["Other"] * n_league
})

# Trim extreme noise (presentation-quality)
df = df[
    (df["Shared Minutes"] >= 30) &
    (df["Net Rating Variability"] <= 8)
]

# ----------------------------
# Precompute trend (binning)
# ----------------------------
bins = np.linspace(30, 900, 20)

trend_df = (
    df[df["Team"] == "Other"]
    .assign(bin=pd.cut(df["Shared Minutes"], bins))
    .groupby("bin", observed=True)
    .agg(
        Minutes=("Shared Minutes", "mean"),
        Volatility=("Net Rating Variability", "mean")
    )
    .dropna()
)

# ----------------------------
# Scatter plot
# ----------------------------
scatter = alt.Chart(df).mark_circle(opacity=0.7).encode(
    x=alt.X(
        "Shared Minutes:Q",
        title="Lineup Stability (Shared Minutes)"
    ),
    y=alt.Y(
        "Net Rating Variability:Q",
        title="Performance Variability"
    ),
    color=alt.Color(
        "Team:N",
        scale=alt.Scale(
            domain=["Bulls", "Other"],
            range=["#CE1141", "#C7C7C7"]
        ),
        legend=alt.Legend(title=None)
    ),
    tooltip=[
        "Shared Minutes",
        "Net Rating Variability",
        "Team"
    ]
)

# ----------------------------
# Trend line
# ----------------------------
trend = alt.Chart(trend_df).mark_line(
    color="black",
    strokeWidth=3
).encode(
    x="Minutes:Q",
    y="Volatility:Q"
)

stable_band = alt.Chart(
    pd.DataFrame({
        "x1": [400],
        "x2": [900]
    })
).mark_rect(
    opacity=0.08,
    color="gray"
).encode(
    x="x1:Q",
    x2="x2:Q"
)


avg_vol = df["Net Rating Variability"].mean()

baseline = alt.Chart(
    pd.DataFrame({"y": [avg_vol]})
).mark_rule(
    strokeDash=[4,4],
    color="black",
    opacity=0.4
).encode(
    y="y:Q"
)

scatter = scatter.encode(
    opacity=alt.condition(
        alt.datum.Team == "Bulls",
        alt.value(1.0),
        alt.value(0.5)
    )
)


# ----------------------------
# Final chart
# ----------------------------
chart = (scatter + trend + stable_band + baseline).properties(
    width=680,
    height=450,
    title="Lineup Stability vs Performance Variability"
)

chart


alt.LayerChart(...)

In [22]:
import numpy as np
import pandas as pd
import altair as alt

# Assume df already exists from Option 2
# Columns:
# "Shared Minutes", "Net Rating Variability", "Team"

def stability_bucket(m):
    if m < 200:
        return "Low Stability"
    elif m < 450:
        return "Medium Stability"
    else:
        return "High Stability"

df["Stability Regime"] = df["Shared Minutes"].apply(stability_bucket)


In [23]:
boxplot = alt.Chart(df).mark_boxplot(
    extent="min-max",
    size=40
).encode(
    x=alt.X(
        "Net Rating Variability:Q",
        title="Performance Variability"
    ),
    y=alt.Y(
        "Stability Regime:N",
        title=None,
        sort=["Low Stability", "Medium Stability", "High Stability"]
    ),
    color=alt.Color(
        "Stability Regime:N",
        legend=None,
        scale=alt.Scale(
            range=["#D3D3D3", "#A9A9A9", "#6E6E6E"]
        )
    )
).properties(
    width=650,
    height=250,
    title="Performance Variability by Lineup Stability Regime"
)


In [24]:
bulls_points = alt.Chart(
    df[df["Team"] == "Bulls"]
).mark_circle(
    size=70,
    color="#CE1141",
    opacity=0.9
).encode(
    x="Net Rating Variability:Q",
    y=alt.Y(
        "Stability Regime:N",
        sort=["Low Stability", "Medium Stability", "High Stability"]
    ),
    tooltip=[
        "Shared Minutes",
        "Net Rating Variability"
    ]
)

chart = boxplot + bulls_points
chart


alt.LayerChart(...)